# Ingest data to Bronze zone

- Spark reads database (postgres) and write (partition by date) to parquet file in Bronze zone

In [1]:
from pyspark.sql import SparkSession
import os
import pyspark.sql.functions as F
from dotenv import load_dotenv

load_dotenv('../../../../.env_file/infra.env'), load_dotenv('../../../../.env_file/app.env')

(True, True)

- Config Spark để thực hiện một số việc sau:
    - Đọc/Ghi trên S3: Cần phải truyền vào credential của AWS tương ứng với account. Việc này có thể gặp một số lỗi do conflict policy trên AWS. Ví dụ: 1 user vừa thuộc group admin (full quyền), vừa thuộc group dev (không có quyền tạo bucket trên S3). Lúc này AWS sẽ ưu tiên các quyền bị chặn. Cần remove user ra khỏi group có quyền bị chặn
    - Đọc/Ghi Postgres: Truyền vào file jar tương ứng
    - Vì tôi đang ingest data vào Bronze zone (which is a data lake architecture), tôi phải đảm bảo các hoạt động của tôi được ghi lại bởi Hive. Cụ thể, tôi cần config endpoint của Hive, đảm bảo Spark tải đúng config version của Hive từ maven, và phải cung cấp một location để lưu thông tin schema (đoạn này thì tôi không rõ những nếu không cung cấp location thì khi tạo schema từ spark, sẽ bị lỗi)

In [2]:
spark = (
    SparkSession.builder
    .appName('ingest data to bronze zone')
    # read/write S3
    .config("spark.hadoop.fs.s3a.access.key", os.getenv('AWS_ACCESS_KEY_ID'))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv('AWS_SECRET_ACCESS_KEY'))
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    # read postgres
    .config(
        'spark.jars',
        """
        ../../../../spark/jar/postgresql-42.7.11.jar,
        ../../../../spark/jar/hadoop-aws-3.4.2.jar,
        ../../../../spark/jar/bundle-2.29.52.jar
        """
    )
    # Hive metastore
    .config("spark.hadoop.hive.metastore.uris", "thrift://localhost:9083")
    .config("spark.sql.catalogImplementation", "hive")
    # create schema using spark (instead of using trino)
    .config("spark.sql.warehouse.dir", "s3a://flezi-data/warehouse/")
    .enableHiveSupport()
    .getOrCreate()
)
spark

26/07/19 14:29:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [3]:
# create schema in trino
spark.sql('create schema if not exists bronze')

DataFrame[]

In [3]:
df = (
    spark.read.format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("url", f"jdbc:postgresql://localhost:{os.getenv('COM_DB_PORT')}/{os.getenv('COM_DB_NAME')}")
    .option("dbtable", "public.nyc_taxi")
    .option("user", os.getenv("COM_DB_USER"))
    .option("password", os.getenv("COM_DB_PASSWORD"))
    .load()
)

In [4]:
watermark = '2000-01-01 00:00:00.0'

df.createOrReplaceTempView('nyc_taxi')
df = spark.sql(f'''
    select * from nyc_taxi
    where source_created_datetime <= (
        select max(source_created_datetime)
        from (
            select * from nyc_taxi
            where source_created_datetime > timestamp '{watermark}'
            order by source_created_datetime asc
            limit {140_000*3}
        )
    ) and source_created_datetime > timestamp '{watermark}';
''').cache()  # cache to avoid re-reading from Postgres on each action

df.show()

26/07/19 02:15:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------+------------+-----------+-----+-------+---------+---------------------+----------+------------+------------+--------------------+-----------+------------------+--------------------+-----------------------+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|     trip_distance|rate_code_id|store_and_fwd_flag|  pickup_longitude|pickup_latitude| dropoff_longitude|dropoff_latitude|pu_location_id|do_location_id|payment_type|fare_amount|extra|mta_tax|surcharge|improvement_surcharge|tip_amount|tolls_amount|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|         source_file|source_created_datetime|
+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+-------------

In [5]:
month_partition_df = (
    df.withColumn('year', F.year('pickup_datetime'))
    .withColumn('month', F.month('pickup_datetime'))
)

In [6]:
S3_BRONZE_PATH = "s3a://flezi-data/bronze/"
SCHEMA = "taxi_data"
TABLE  = "nyc_taxi"

(
    month_partition_df
    .write
    .partitionBy("year", "month")
    .mode("append")
    .option("path", S3_BRONZE_PATH)
    .saveAsTable(f"{SCHEMA}.{TABLE}")   # đăng ký luôn với Hive metastore
)

print(f"Done. Table registered as '{SCHEMA}.{TABLE}' → {S3_BRONZE_PATH}")

26/05/29 09:48:03 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://flezi-data/bronze.
java.io.FileNotFoundException: No such file or directory: s3a://flezi-data/bronze
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:4169)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:4027)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$getFileStatus$21(S3AFileSystem.java:4004)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.invokeTrackingDuration(IOStatisticsBinding.java:547)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.lambda$trackDurationOfOperation$5(IOStatisticsBinding.java:528)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.trackDuration(IOStatisticsBinding.java:449)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.trackDurationAndSpan(S3AFileSystem.java:2917)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.trackDurationAndSpan(S3AFileS

Done. Table registered as 'taxi_data.nyc_taxi' → s3a://flezi-data/bronze/


In [4]:
latest_pickup_date = df.select(F.max('pickup_datetime')).collect()
latest_pickup_date

[Row(max(pickup_datetime)=datetime.datetime(2009, 1, 31, 23, 59, 55))]

In [5]:
str(latest_pickup_date[0][0])

'2009-01-31 23:59:55'

---

In [28]:
latest_pickup_date = df.select(F.max('pickup_datetime')).collect()
latest_pickup_date

[Row(max(pickup_datetime)=None)]

In [ ]:
df = (
    spark.read.format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("url", f"jdbc:postgresql://localhost:{os.getenv('COM_DB_PORT')}/{os.getenv('COM_DB_NAME')}")
    .option("dbtable", "public.nyc_taxi")
    .option("user", os.getenv("COM_DB_USER"))
    .option("password", os.getenv("COM_DB_PASSWORD"))
    .load()
)

# df = df.filter(
#     df['pickup_datetime'] > str(latest_pickup_date[0][0])
# )

df = df.filter(
    df['pickup_datetime'] > '2009-08-28'
)

if df.count() > 0:
    print('continue process')
else:
    print('return previous last date')

157066

In [ ]:
month_partition_df = (
    df.withColumn('year', F.year('pickup_datetime'))
    .withColumn('month', F.month('pickup_datetime'))
)

(
    month_partition_df
    .write
    .partitionBy("year", "month")
    .mode("append")
    .option("path", S3_BRONZE_PATH)
    .saveAsTable(f"{SCHEMA}.{TABLE}")   # đăng ký luôn với Hive metastore
)

print(f"Done. Table registered as '{SCHEMA}.{TABLE}' → {S3_BRONZE_PATH}")

Done. Table registered as 'taxi_data.nyc_taxi' → s3a://flezi-data/bronze/
